# **Convolución 2D**

Este cuaderno explora la operación de convolución 2D. Aquí escribirás la convolución a mano para comprobar que calculamos lo mismo que PyTorch.

Ejecuta las celdas en orden. En varios lugares verás la palabra "TAREA": sigue esas instrucciones, anticipa qué ocurrirá o escribe el código necesario para completar las funciones.

In [ ]:
import numpy as np
import torch
# Configura una impresión legible
np.set_printoptions(precision=3, floatmode="fixed")
torch.set_printoptions(precision=3)

Esta rutina realiza una convolución en PyTorch

In [ ]:
# Realiza la convolución en PyTorch
def conv_pytorch(image, conv_weights, stride=1, pad =1):
  # Convierte la imagen y el kernel en tensores
  image_tensor = torch.from_numpy(image) # (tamaño_lote, canales_entrada, alto_entrada, ancho_entrada)
  conv_weights_tensor = torch.from_numpy(conv_weights) # (canales_salida, canales_entrada, alto_kernel, ancho_kernel)
  # Ejecuta la convolución
  output_tensor = torch.nn.functional.conv2d(image_tensor, conv_weights_tensor, stride=stride, padding=pad)
  # Convierte de vuelta desde PyTorch y devuelve el resultado
  return(output_tensor.numpy()) # (tamaño_lote, canales_salida, alto_salida, ancho_salida)

Primero empezaremos con la convolución 2D más simple: un canal de entrada, un canal de salida y una sola imagen en el lote.

In [ ]:
# Realiza la convolución en NumPy
def conv_numpy_1(image, weights, pad=1):

    # Aplica relleno de ceros
    if pad != 0:
        image = np.pad(image, ((0, 0), (0 ,0), (pad, pad), (pad, pad)),'constant')

    # Obtiene los tamaños del arreglo de imagen y de los pesos del kernel
    batchSize,  channelsIn, imageHeightIn, imageWidthIn = image.shape
    channelsOut, channelsIn, kernelHeight, kernelWidth = weights.shape

    # Obtiene el tamaño de los arreglos de salida
    imageHeightOut = np.floor(1 + imageHeightIn - kernelHeight).astype(int)
    imageWidthOut = np.floor(1 + imageWidthIn - kernelWidth).astype(int)

    # Crea la salida
    out = np.zeros((batchSize, channelsOut, imageHeightOut, imageWidthOut), dtype=np.float32)

    # !!!!!! NOTA: HAY UN DETALLE IMPORTANTE AQUÍ !!!!!!!!
    # La imagen se rellenó con ceros, así que queda rodeada por un "anillo" de ceros
    # Eso significa que los índices de la imagen quedan desplazados por uno
    # Esto en realidad simplifica el código

    for c_y in range(imageHeightOut):
      for c_x in range(imageWidthOut):
        for c_kernel_y in range(kernelHeight):
          for c_kernel_x in range(kernelWidth):
            # TAREA -- recupera el píxel de la imagen y el peso de la convolución
            # Solo hay una imagen en el lote, un canal de entrada y un canal de salida, así que estos índices deben ser cero
            # Reemplaza las dos líneas siguientes
            this_pixel_value = image[0, 0, c_y + c_kernel_y, c_x + c_kernel_x]
            this_weight = weights[0, 0, c_kernel_y, c_kernel_x]


            # Multiplica estos valores y súmalos a la salida en esta posición
            out[0, 0, c_y, c_x] += np.sum(this_pixel_value * this_weight)

    return out

In [ ]:
# Fija la semilla aleatoria para obtener siempre la misma respuesta
np.random.seed(1)
n_batch = 1
image_height = 4
image_width = 6
channels_in = 1
kernel_size = 3
channels_out = 1

# Crea una imagen de entrada aleatoria
input_image= np.random.normal(size=(n_batch, channels_in, image_height, image_width))
# Crea pesos aleatorios para el kernel de convolución
conv_weights = np.random.normal(size=(channels_out, channels_in, kernel_size, kernel_size))

# Realiza la convolución usando PyTorch
conv_results_pytorch = conv_pytorch(input_image, conv_weights, stride=1, pad=1)
print("Resultados de PyTorch")
print(conv_results_pytorch)

# Realiza la convolución en NumPy
print("Tus resultados")
conv_results_numpy = conv_numpy_1(input_image, conv_weights)
print(conv_results_numpy)

print("Mean Absolute Error PyTorch y NumPy")
print(np.mean(np.abs(conv_results_pytorch - conv_results_numpy)))

Ahora añadamos la posibilidad de usar distintos pasos

In [ ]:
# Realiza la convolución en NumPy
def conv_numpy_2(image, weights, stride=1, pad=1):

    # Aplica relleno de ceros
    if pad != 0:
        image = np.pad(image, ((0, 0), (0 ,0), (pad, pad), (pad, pad)),'constant')

    # Obtiene los tamaños del arreglo de imagen y de los pesos del kernel
    batchSize,  channelsIn, imageHeightIn, imageWidthIn = image.shape
    channelsOut, channelsIn, kernelHeight, kernelWidth = weights.shape

    # Obtiene el tamaño de los arreglos de salida
    imageHeightOut = np.floor(1 + (imageHeightIn - kernelHeight) / stride).astype(int)
    imageWidthOut = np.floor(1 + (imageWidthIn - kernelWidth) / stride).astype(int)

    # Crea la salida
    out = np.zeros((batchSize, channelsOut, imageHeightOut, imageWidthOut), dtype=np.float32)

    for c_y in range(imageHeightOut):
      for c_x in range(imageWidthOut):
        for c_kernel_y in range(kernelHeight):
          for c_kernel_x in range(kernelWidth):
            # TAREA -- recupera el píxel de la imagen y el peso de la convolución
            # Solo hay una imagen en el lote, un canal de entrada y un canal de salida, así que estos índices deben ser cero
            # Reemplaza las dos líneas siguientes
            this_pixel_value = image[0, 0, c_y * stride + c_kernel_y, c_x * stride + c_kernel_x]
            this_weight = weights[0,0, c_kernel_y, c_kernel_x]


            # Multiplica estos valores y súmalos a la salida en esta posición
            out[0, 0, c_y, c_x] += np.sum(this_pixel_value * this_weight)

    return out

In [ ]:
# Fija la semilla aleatoria para obtener siempre la misma respuesta
np.random.seed(1)
n_batch = 1
image_height = 12
image_width = 10
channels_in = 1
kernel_size = 3
channels_out = 1
stride = 2

# Crea una imagen de entrada aleatoria
input_image= np.random.normal(size=(n_batch, channels_in, image_height, image_width))
# Crea pesos aleatorios para el kernel de convolución
conv_weights = np.random.normal(size=(channels_out, channels_in, kernel_size, kernel_size))

# Realiza la convolución usando PyTorch
conv_results_pytorch = conv_pytorch(input_image, conv_weights, stride, pad=1)
print("Resultados de PyTorch")
print(conv_results_pytorch)

# Realiza la convolución en NumPy
print("Tus resultados")
conv_results_numpy = conv_numpy_2(input_image, conv_weights, stride, pad=1)
print(conv_results_numpy)

print("Mean Absolute Error PyTorch y NumPy")
print(np.mean(np.abs(conv_results_pytorch - conv_results_numpy)))

Ahora introduciremos varios canales de entrada y de salida

In [ ]:
# Realiza la convolución en NumPy
def conv_numpy_3(image, weights, stride=1, pad=1):

    # Aplica relleno de ceros
    if pad != 0:
        image = np.pad(image, ((0, 0), (0 ,0), (pad, pad), (pad, pad)),'constant')

    # Obtiene los tamaños del arreglo de imagen y de los pesos del kernel
    batchSize,  channelsIn, imageHeightIn, imageWidthIn = image.shape
    channelsOut, channelsIn, kernelHeight, kernelWidth = weights.shape

    # Obtiene el tamaño de los arreglos de salida
    imageHeightOut = np.floor(1 + (imageHeightIn - kernelHeight) / stride).astype(int)
    imageWidthOut = np.floor(1 + (imageWidthIn - kernelWidth) / stride).astype(int)

    # Crea la salida
    out = np.zeros((batchSize, channelsOut, imageHeightOut, imageWidthOut), dtype=np.float32)

    for c_y in range(imageHeightOut):
      for c_x in range(imageWidthOut):
        for c_channel_out in range(channelsOut):
          for c_channel_in in range(channelsIn):
            for c_kernel_y in range(kernelHeight):
              for c_kernel_x in range(kernelWidth):
                  # TAREA -- recupera el píxel de la imagen y el peso de la convolución
                  # Solo hay una imagen en el lote, así que este índice debe ser cero
                  # Reemplaza las dos líneas siguientes
                  this_pixel_value = image[0, c_channel_in, c_y * stride + c_kernel_y, c_x * stride + c_kernel_x]
                  this_weight = weights[c_channel_out, c_channel_in, c_kernel_y, c_kernel_x]

                  # Multiplica estos valores y súmalos a la salida en esta posición
                  out[0, c_channel_out, c_y, c_x] += np.sum(this_pixel_value * this_weight)
    return out

In [ ]:
# Fija la semilla aleatoria para obtener siempre la misma respuesta
np.random.seed(1)
n_batch = 1
image_height = 4
image_width = 6
channels_in = 5
kernel_size = 3
channels_out = 2

# Crea una imagen de entrada aleatoria
input_image= np.random.normal(size=(n_batch, channels_in, image_height, image_width))
# Crea pesos aleatorios para el kernel de convolución
conv_weights = np.random.normal(size=(channels_out, channels_in, kernel_size, kernel_size))

# Realiza la convolución usando PyTorch
conv_results_pytorch = conv_pytorch(input_image, conv_weights, stride=1, pad=1)
print("Resultados de PyTorch")
print(conv_results_pytorch)

# Realiza la convolución en NumPy
print("Tus resultados")
conv_results_numpy = conv_numpy_3(input_image, conv_weights, stride=1, pad=1)
print(conv_results_numpy)

print("Mean Absolute Error PyTorch y NumPy")
print(np.mean(np.abs(conv_results_pytorch - conv_results_numpy)))

Ahora haremos la convolución completa con varias imágenes (tamaño de lote > 1), varios canales de entrada y varios canales de salida.

In [ ]:
# Realiza la convolución en NumPy
def conv_numpy_4(image, weights, stride=1, pad=1):

    # Aplica relleno de ceros
    if pad != 0:
        image = np.pad(image, ((0, 0), (0 ,0), (pad, pad), (pad, pad)),'constant')

    # Obtiene los tamaños del arreglo de imagen y de los pesos del kernel
    batchSize,  channelsIn, imageHeightIn, imageWidthIn = image.shape
    channelsOut, channelsIn, kernelHeight, kernelWidth = weights.shape

    # Obtiene el tamaño de los arreglos de salida
    imageHeightOut = np.floor(1 + (imageHeightIn - kernelHeight) / stride).astype(int)
    imageWidthOut = np.floor(1 + (imageWidthIn - kernelWidth) / stride).astype(int)

    # Crea la salida
    out = np.zeros((batchSize, channelsOut, imageHeightOut, imageWidthOut), dtype=np.float32)

    for c_batch in range(batchSize):
      for c_y in range(imageHeightOut):
        for c_x in range(imageWidthOut):
          for c_channel_out in range(channelsOut):
            for c_channel_in in range(channelsIn):
              for c_kernel_y in range(kernelHeight):
                for c_kernel_x in range(kernelWidth):
                    # TAREA -- recupera el píxel de la imagen y el peso de la convolución
                    # Reemplaza las dos líneas siguientes
                    this_pixel_value = image[c_batch, c_channel_in, c_y * stride + c_kernel_y, c_x * stride + c_kernel_x]
                    this_weight = weights[c_channel_out, c_channel_in, c_kernel_y, c_kernel_x]



                    # Multiplica estos valores y súmalos a la salida en esta posición
                    out[c_batch, c_channel_out, c_y, c_x] += np.sum(this_pixel_value * this_weight)
    return out

In [ ]:
# Fija la semilla aleatoria para obtener siempre la misma respuesta
np.random.seed(1)
n_batch = 2
image_height = 4
image_width = 6
channels_in = 5
kernel_size = 3
channels_out = 2

# Crea una imagen de entrada aleatoria
input_image= np.random.normal(size=(n_batch, channels_in, image_height, image_width))
# Crea pesos aleatorios para el kernel de convolución
conv_weights = np.random.normal(size=(channels_out, channels_in, kernel_size, kernel_size))

# Realiza la convolución usando PyTorch
conv_results_pytorch = conv_pytorch(input_image, conv_weights, stride=1, pad=1)
print("Resultados de PyTorch")
print(conv_results_pytorch)

# Realiza la convolución en NumPy
print("Tus resultados")
conv_results_numpy = conv_numpy_4(input_image, conv_weights, stride=1, pad=1)
print(conv_results_numpy)

print("Mean Absolute Error PyTorch y NumPy")
print(np.mean(np.abs(conv_results_pytorch - conv_results_numpy)))

## Paralelismo en GPU para convoluciones 2D

Hasta ahora implementamos la convolución con ciclos anidados para entender qué operación se está calculando. Ese enfoque es útil para aprender, pero no es eficiente: cada salida se calcula una por una en Python.

En una CNN real, una convolución multicanal produce un tensor de salida con muchas posiciones independientes. Para cada imagen del lote, cada canal de salida y cada posición espacial se calcula:

$$
y_{n,c_o,i,j} = \sum_{c_i=0}^{C_i-1}\sum_{u=0}^{K_h-1}\sum_{v=0}^{K_w-1}
x_{n,c_i,i+u,j+v}\, w_{c_o,c_i,u,v}
$$

El valor $y_{n,c_o,i,j}$ no necesita esperar a que se calcule $y_{n,c_o,i,j+1}$ ni el de otro canal de salida. Por eso el trabajo se puede repartir en paralelo sobre varias dimensiones:

- distintas imágenes del lote `n`,
- distintos canales de salida `c_o`,
- distintas posiciones espaciales `(i, j)`.

CUDA aprovecha esta independencia asignando muchos de esos cálculos a miles de hilos de la GPU. Cada hilo o grupo de hilos calcula una parte del tensor de salida, mientras la suma sobre canales de entrada y posiciones del kernel se realiza con operaciones optimizadas. Por eso una convolución grande, con muchos canales y muchas posiciones espaciales, suele ser mucho más rápida en GPU que en CPU.

En PyTorch no escribimos los hilos CUDA manualmente. Movemos los tensores al dispositivo adecuado con `.cuda()` y usamos la misma función `torch.nn.functional.conv2d`. Si no hay CUDA, la celda también revisa `mps`, que es el backend de Metal usado en Apple Silicon.

La siguiente comparación no busca entrenar una red; solo mide cuánto tarda una convolución 2D grande en CPU y en GPU cuando el hardware está disponible.


In [1]:
# Comparación de una convolución 2D grande en CPU, CUDA y MPS si están disponibles
import time
import torch
import torch.nn.functional as F


def sincronizar(device):
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps" and hasattr(torch, "mps"):
        torch.mps.synchronize()


def medir_conv2d(x, w, device, repeticiones=3, calentamiento=1):
    x_dev = x.to(device)
    w_dev = w.to(device)

    with torch.no_grad():
        for _ in range(calentamiento):
            y = F.conv2d(x_dev, w_dev, stride=1, padding=1)
        sincronizar(device)

        inicio = time.perf_counter()
        for _ in range(repeticiones):
            y = F.conv2d(x_dev, w_dev, stride=1, padding=1)
        sincronizar(device)
        tiempo_promedio = (time.perf_counter() - inicio) / repeticiones

    return tiempo_promedio, y


# Tamaño grande pero ajustable. Si la CPU tarda demasiado, reduce image_size o los canales.
torch.manual_seed(0)
n_batch = 8
channels_in = 32
channels_out = 64
image_size = 128
kernel_size = 3

x = torch.randn(n_batch, channels_in, image_size, image_size)
w = torch.randn(channels_out, channels_in, kernel_size, kernel_size)

print(f"Entrada: {tuple(x.shape)}")
print("Formato: (n_batch, channels, height, width)")
print(f"Kernel:  {tuple(w.shape)}")
print("Formato del kernel: (channels_out, channels_in, kernel_height, kernel_width)")
print(f"Salida esperada: ({n_batch}, {channels_out}, {image_size}, {image_size})")

cpu = torch.device("cpu")
x_cpu = x.cpu()
w_cpu = w.cpu()
tiempo_cpu, y_cpu = medir_conv2d(x_cpu, w_cpu, cpu, repeticiones=1, calentamiento=1)
print(f"CPU:  {tiempo_cpu:.4f} segundos por convolución")

if torch.cuda.is_available():
    cuda = torch.device("cuda")
    x_cuda = x.cuda()
    w_cuda = w.cuda()
    tiempo_cuda, y_cuda = medir_conv2d(x_cuda, w_cuda, cuda, repeticiones=5, calentamiento=2)
    diferencia = (y_cpu - y_cuda.cpu()).abs().max().item()
    print(f"CUDA: {tiempo_cuda:.4f} segundos por convolución")
    print(f"Aceleración CPU/CUDA: {tiempo_cpu / tiempo_cuda:.2f}x")
    print(f"Diferencia máxima CPU vs CUDA: {diferencia:.6f}")
else:
    print("CUDA no está disponible en este equipo.")

    mps_disponible = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
    if mps_disponible:
        mps = torch.device("mps")
        tiempo_mps, y_mps = medir_conv2d(x, w, mps, repeticiones=5, calentamiento=2)
        diferencia = (y_cpu - y_mps.cpu()).abs().max().item()
        print(f"MPS/Metal: {tiempo_mps:.4f} segundos por convolución")
        print(f"Aceleración CPU/MPS: {tiempo_cpu / tiempo_mps:.2f}x")
        print(f"Diferencia máxima CPU vs MPS: {diferencia:.6f}")
    else:
        print("Tampoco está disponible MPS/Metal; solo se midió CPU.")


Entrada: (8, 32, 128, 128)
Formato: (n_batch, channels, height, width)
Kernel:  (64, 32, 3, 3)
Formato del kernel: (channels_out, channels_in, kernel_height, kernel_width)
Salida esperada: (8, 64, 128, 128)
CPU:  0.0180 segundos por convolución
CUDA: 0.0012 segundos por convolución
Aceleración CPU/CUDA: 14.45x
Diferencia máxima CPU vs CUDA: 0.030002
